# Market Basket Analysis and Product Recommendation System using Online Retail Data

## Project Overview

In the previous projects, the focus was on customer segmentation and churn prediction.  
In this project, the focus shifts towards understanding product purchase patterns from transaction-level data.

The main idea is to study which products are frequently bought together and how those relationships can help in building recommendation logic for cross-selling and bundling.

This project will also make use of SQL Server for structured data storage and Power BI for dashboarding in the later stages.

## Objective

The objective of this project is to analyze online retail transactions and prepare the data for market basket analysis.

The work in this notebook focuses on:
- loading and understanding the dataset
- checking the quality of the data
- identifying cancelled or invalid transactions
- cleaning the data for product-level analysis
- creating a transaction dataset that can later be used for association rule mining

In [2]:
import pandas as pd
import numpy as np
import pyarrow

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("pyarrow:", pyarrow.__version__)

pandas: 3.0.1
numpy: 2.4.3
pyarrow: 23.0.1


In [6]:
df = pd.read_excel("Online Retail Dataset.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (541909, 8)

Columns:
 ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Missing values:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows: 5268


In [8]:
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

In [9]:
print("Cancelled invoice rows:", df["InvoiceNo"].astype(str).str.startswith("C").sum())
print("Rows with non-positive quantity:", (df["Quantity"] <= 0).sum())
print("Rows with non-positive unit price:", (df["UnitPrice"] <= 0).sum())

Cancelled invoice rows: 9288
Rows with non-positive quantity: 10624
Rows with non-positive unit price: 2517


In [10]:
basket_df = df.copy()

basket_df = basket_df.dropna(subset=["Description"])
basket_df = basket_df[~basket_df["InvoiceNo"].astype(str).str.startswith("C")]
basket_df = basket_df[basket_df["Quantity"] > 0]
basket_df = basket_df[basket_df["UnitPrice"] > 0]

In [11]:
print("Original shape:", df.shape)
print("Cleaned shape:", basket_df.shape)

Original shape: (541909, 9)
Cleaned shape: (530104, 9)


In [12]:
print("Unique invoices:", basket_df["InvoiceNo"].nunique())
print("Unique products:", basket_df["Description"].nunique())
print("Unique customers:", basket_df["CustomerID"].nunique())
print("Unique countries:", basket_df["Country"].nunique())

Unique invoices: 19960
Unique products: 4026
Unique customers: 4338
Unique countries: 38


In [13]:
top_products = (
    basket_df.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        78033
WORLD WAR 2 GLIDERS ASSTD DESIGNS     55047
JUMBO BAG RED RETROSPOT               48474
WHITE HANGING HEART T-LIGHT HOLDER    37891
POPCORN HOLDER                        36761
ASSORTED COLOUR BIRD ORNAMENT         36461
PACK OF 72 RETROSPOT CAKE CASES       36419
RABBIT NIGHT LIGHT                    30788
MINI PAINT SET VINTAGE                26633
Name: Quantity, dtype: int64

In [14]:
basket_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## Summary

In this step, the Online Retail dataset was loaded and checked quickly for its structure, missing values, duplicate rows, and invalid transaction records. A new column called `TotalPrice` was created using quantity and unit price.

Since the project focuses on market basket analysis, only valid completed transactions were kept. Rows with missing product descriptions, cancelled invoices, non-positive quantities, and non-positive unit prices were removed. The cleaned dataset is now ready for the next stage, where invoice-level product baskets will be created for association analysis and recommendation logic.

In [15]:
country_counts = basket_df["Country"].value_counts()
country_counts.head(10)

Country
United Kingdom    485123
Germany             9040
France              8407
EIRE                7890
Spain               2484
Netherlands         2359
Belgium             2031
Switzerland         1966
Portugal            1501
Australia           1182
Name: count, dtype: int64

## Note on Country Selection

Before creating the basket data, I checked the number of transaction records for each country using the `Country` column. The United Kingdom had by far the highest number of records in the cleaned dataset.

Because of this, the basket analysis is focused on United Kingdom transactions. Restricting the data to the dominant market helps reduce sparsity and makes the product association patterns more stable and meaningful. It also keeps the analysis more efficient compared to using many countries with much smaller transaction volumes.

In [16]:
uk_df = basket_df[basket_df["Country"] == "United Kingdom"]

print("UK data shape:", uk_df.shape)
print("Unique UK invoices:", uk_df["InvoiceNo"].nunique())
print("Unique UK products:", uk_df["Description"].nunique())

UK data shape: (485123, 9)
Unique UK invoices: 18019
Unique UK products: 4007


In [17]:
basket = (
    uk_df.groupby(["InvoiceNo", "Description"])["Quantity"]
    .sum()
    .unstack()
    .fillna(0)
)

basket.head()

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536366,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536367,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536368,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536369,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
print("Basket shape:", basket.shape)

Basket shape: (18019, 4007)


In [19]:
def encode_units(x):
    if x <= 0:
        return 0
    else:
        return 1

In [20]:
basket_encoded = basket.map(encode_units)
basket_encoded.head()

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [21]:
print("Encoded basket shape:", basket_encoded.shape)

Encoded basket shape: (18019, 4007)


In [22]:
basket_encoded.iloc[:5, :10]

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT
InvoiceNo,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0


In [23]:
product_support_count = basket_encoded.sum().sort_values(ascending=False)
product_support_count.head(10)

Description
WHITE HANGING HEART T-LIGHT HOLDER    2162
JUMBO BAG RED RETROSPOT               1935
REGENCY CAKESTAND 3 TIER              1685
PARTY BUNTING                         1593
LUNCH BAG RED RETROSPOT               1392
ASSORTED COLOUR BIRD ORNAMENT         1371
SET OF 3 CAKE TINS PANTRY DESIGN      1241
NATURAL SLATE HEART CHALKBOARD        1219
LUNCH BAG  BLACK SKULL.               1216
HEART OF WICKER SMALL                 1164
dtype: int64

## Summary

In this step, the cleaned transaction data was filtered to include only United Kingdom records, since this country had the highest number of transactions in the dataset.

Then, the data was transformed into an invoice-product basket format, where each row represents an invoice and each column represents a product. The quantity values were converted into binary form so that each product is marked as either present or absent in a basket.

This encoded basket data is now ready for the next stage of the project, where frequent itemsets and association rules will be generated.

In [24]:
!pip install mlxtend

from mlxtend.frequent_patterns import apriori

Defaulting to user installation because normal site-packages is not writeable


In [25]:
basket_encoded = basket.map(encode_units).astype(bool)
basket_encoded.head()

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536366,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [26]:
frequent_itemsets = apriori(basket_encoded, min_support=0.02, use_colnames=True)
frequent_itemsets.head()

,support,itemsets
0,0.020145,frozenset({3 HOOK PHOTO SHELF ANTIQUE WHITE})
1,0.024252,frozenset({3 STRIPEY MICE FELTCRAFT})
2,0.021810,frozenset({4 TRADITIONAL SPINNING TOPS})
3,0.047450,frozenset({6 RIBBONS RUSTIC CHARM})
4,0.020589,frozenset({60 CAKE CASES DOLLY GIRL DESIGN})


In [27]:
print("Frequent itemsets shape:", frequent_itemsets.shape)

Frequent itemsets shape: (400, 2)


In [28]:
requent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False)
frequent_itemsets.head(10)

,support,itemsets
0,0.020145,frozenset({3 HOOK PHOTO SHELF ANTIQUE WHITE})
1,0.024252,frozenset({3 STRIPEY MICE FELTCRAFT})
2,0.021810,frozenset({4 TRADITIONAL SPINNING TOPS})
3,0.047450,frozenset({6 RIBBONS RUSTIC CHARM})
4,0.020589,frozenset({60 CAKE CASES DOLLY GIRL DESIGN})
5,0.032244,frozenset({60 CAKE CASES VINTAGE CHRISTMAS})
6,0.041789,frozenset({60 TEATIME FAIRY CAKE CASES})
7,0.030801,frozenset({72 SWEETHEART FAIRY CAKE CASES})
8,0.020700,frozenset({AIRLINE BAG VINTAGE TOKYO 78})
9,0.021089,frozenset({ALARM CLOCK BAKELIKE CHOCOLATE})


In [29]:
frequent_itemsets["itemset_length"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets.head()

,support,itemsets,itemset_length
0,0.020145,frozenset({3 HOOK PHOTO SHELF ANTIQUE WHITE}),1
1,0.024252,frozenset({3 STRIPEY MICE FELTCRAFT}),1
2,0.021810,frozenset({4 TRADITIONAL SPINNING TOPS}),1
3,0.047450,frozenset({6 RIBBONS RUSTIC CHARM}),1
4,0.020589,frozenset({60 CAKE CASES DOLLY GIRL DESIGN}),1


In [30]:
frequent_itemsets["itemset_length"].value_counts().sort_index()

itemset_length
1    299
2     97
3      4
Name: count, dtype: int64

In [31]:
frequent_itemsets[frequent_itemsets["itemset_length"] == 1].head(10)

,support,itemsets,itemset_length
0,0.020145,frozenset({3 HOOK PHOTO SHELF ANTIQUE WHITE}),1
1,0.024252,frozenset({3 STRIPEY MICE FELTCRAFT}),1
2,0.021810,frozenset({4 TRADITIONAL SPINNING TOPS}),1
3,0.047450,frozenset({6 RIBBONS RUSTIC CHARM}),1
4,0.020589,frozenset({60 CAKE CASES DOLLY GIRL DESIGN}),1
5,0.032244,frozenset({60 CAKE CASES VINTAGE CHRISTMAS}),1
6,0.041789,frozenset({60 TEATIME FAIRY CAKE CASES}),1
7,0.030801,frozenset({72 SWEETHEART FAIRY CAKE CASES}),1
8,0.020700,frozenset({AIRLINE BAG VINTAGE TOKYO 78}),1
9,0.021089,frozenset({ALARM CLOCK BAKELIKE CHOCOLATE}),1


In [32]:
frequent_itemsets[frequent_itemsets["itemset_length"] == 2].head(10)

,support,itemsets,itemset_length
299,0.022698,"frozenset({60 TEATIME FAIRY CAKE CASES, PACK O...",2
300,0.031245,"frozenset({ALARM CLOCK BAKELIKE RED , ALARM CL...",2
301,0.021810,"frozenset({ALARM CLOCK BAKELIKE RED , ALARM CL...",2
302,0.021311,"frozenset({CHARLOTTE BAG SUKI DESIGN, CHARLOTT...",2
303,0.026583,"frozenset({CHARLOTTE BAG PINK POLKADOT, RED RE...",2
304,0.020090,"frozenset({STRAWBERRY CHARLOTTE BAG, CHARLOTTE...",2
305,0.026361,"frozenset({CHARLOTTE BAG SUKI DESIGN, RED RETR...",2
306,0.021810,"frozenset({CHARLOTTE BAG SUKI DESIGN, STRAWBER...",2
307,0.022920,"frozenset({WOODLAND CHARLOTTE BAG, CHARLOTTE B...",2
308,0.021033,"frozenset({HOT WATER BOTTLE I AM SO POORLY, CH...",2


In [33]:
frequent_itemsets[frequent_itemsets["itemset_length"] >= 3].head(10)

,support,itemsets,itemset_length
396,0.027305,"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",3
397,0.020312,"frozenset({JUMBO BAG PINK POLKADOT, JUMBO BAG ...",3
398,0.022476,"frozenset({JUMBO BAG RED RETROSPOT, JUMBO STOR...",3
399,0.021311,"frozenset({JUMBO BAG RED RETROSPOT, JUMBO STOR...",3


In [34]:
top_2_itemsets = frequent_itemsets[frequent_itemsets["itemset_length"] == 2].sort_values(by="support", ascending=False)
top_2_itemsets.head(10)

,support,itemsets,itemset_length
332,0.043565,"frozenset({JUMBO BAG RED RETROSPOT, JUMBO BAG ...",2
317,0.038848,"frozenset({ROSES REGENCY TEACUP AND SAUCER , G...",2
345,0.038737,"frozenset({JUMBO BAG RED RETROSPOT, JUMBO STOR...",2
343,0.036462,"frozenset({JUMBO BAG RED RETROSPOT, JUMBO SHOP...",2
358,0.033687,"frozenset({LUNCH BAG RED RETROSPOT, LUNCH BAG ...",2
315,0.031966,"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",2
322,0.031633,"frozenset({JUMBO BAG RED RETROSPOT, JUMBO BAG...",2
300,0.031245,"frozenset({ALARM CLOCK BAKELIKE RED , ALARM CL...",2
369,0.030745,"frozenset({LUNCH BAG RED RETROSPOT, LUNCH BAG ...",2
385,0.030246,"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",2


## Summary

In this step, the Apriori algorithm was applied to the encoded basket data to identify frequent itemsets. The `min_support` threshold was used to keep only those product combinations that appear in a meaningful proportion of invoices.

After generating the frequent itemsets, the results were sorted by support and the number of items in each itemset was examined. This helped separate single-product, two-product, and larger combinations before moving to association rule generation in the next stage.

. A minimum support threshold of 2% was used, which means only those item combinations appearing in at least 2% of all baskets were retained.

The analysis produced 400 frequent itemsets in total. Among them, 299 were single-product itemsets, 97 were two-product itemsets, and 4 were three-product itemsets. This shows that while many individual products appear frequently in transactions, a smaller number of product pairs and only a few three-product combinations occur often enough to be considered strong recurring patterns.

The frequent itemsets were then sorted by support to highlight the most common products and product combinations. These results form the basis for the next stage of the project, where association rules will be generated to study directional relationships between products and support recommendation or cross-selling insights.

In [35]:
from mlxtend.frequent_patterns import association_rules

In [36]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({PACK OF 72 RETROSPOT CAKE CASES}),frozenset({60 TEATIME FAIRY CAKE CASES}),0.062656,0.041789,0.022698,0.362267,8.668922,1.0,0.020080,1.502528,0.943779,0.277665,0.334455,0.452714
1,frozenset({60 TEATIME FAIRY CAKE CASES}),frozenset({PACK OF 72 RETROSPOT CAKE CASES}),0.041789,0.062656,0.022698,0.543161,8.668922,1.0,0.020080,2.051802,0.923226,0.277665,0.512624,0.452714
2,frozenset({ALARM CLOCK BAKELIKE RED }),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.051612,0.048615,0.031245,0.605376,12.452370,1.0,0.028736,2.410866,0.969745,0.452936,0.585211,0.624035
3,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE RED }),0.048615,0.051612,0.031245,0.642694,12.452370,1.0,0.028736,2.654274,0.966690,0.452936,0.623249,0.624035
4,frozenset({ALARM CLOCK BAKELIKE RED }),frozenset({ALARM CLOCK BAKELIKE PINK}),0.051612,0.036406,0.021810,0.422581,11.607440,1.0,0.019931,1.668794,0.963581,0.329422,0.400765,0.510833


In [37]:
print("Association rules shape:", rules.shape)
print("\nColumns in rules dataframe:")
print(rules.columns.tolist())

Association rules shape: (184, 14)

Columns in rules dataframe:
['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski']


In [38]:
rules = rules.sort_values(by="confidence", ascending=False)
rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

,antecedents,consequents,support,confidence,lift
163,"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.027305,0.902752,17.453534
165,"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.027305,0.854167,16.099612
31,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.031966,0.820513,15.863541
171,"frozenset({JUMBO SHOPPER VINTAGE RED PAISLEY, ...",frozenset({JUMBO BAG RED RETROSPOT}),0.020312,0.806167,7.507147
176,"frozenset({JUMBO STORAGE BAG SUKI, JUMBO BAG P...",frozenset({JUMBO BAG RED RETROSPOT}),0.022476,0.801980,7.468156
146,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.030246,0.776353,14.632960
162,frozenset({WOODEN STAR CHRISTMAS SCANDINAVIAN}),frozenset({WOODEN HEART CHRISTMAS SCANDINAVIAN}),0.020423,0.768267,27.197263
35,frozenset({GREEN REGENCY TEACUP AND SAUCER}),frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.038848,0.751073,14.156468
181,"frozenset({JUMBO STORAGE BAG SUKI, JUMBO SHOPP...",frozenset({JUMBO BAG RED RETROSPOT}),0.021311,0.748538,6.970494
34,frozenset({ROSES REGENCY TEACUP AND SAUCER }),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.038848,0.732218,14.156468


In [39]:
strong_rules = rules[(rules["confidence"] >= 0.5) & (rules["lift"] > 1.2)]
strong_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

,antecedents,consequents,support,confidence,lift
163,"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.027305,0.902752,17.453534
165,"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.027305,0.854167,16.099612
31,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.031966,0.820513,15.863541
171,"frozenset({JUMBO SHOPPER VINTAGE RED PAISLEY, ...",frozenset({JUMBO BAG RED RETROSPOT}),0.020312,0.806167,7.507147
176,"frozenset({JUMBO STORAGE BAG SUKI, JUMBO BAG P...",frozenset({JUMBO BAG RED RETROSPOT}),0.022476,0.801980,7.468156
146,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.030246,0.776353,14.632960
162,frozenset({WOODEN STAR CHRISTMAS SCANDINAVIAN}),frozenset({WOODEN HEART CHRISTMAS SCANDINAVIAN}),0.020423,0.768267,27.197263
35,frozenset({GREEN REGENCY TEACUP AND SAUCER}),frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.038848,0.751073,14.156468
181,"frozenset({JUMBO STORAGE BAG SUKI, JUMBO SHOPP...",frozenset({JUMBO BAG RED RETROSPOT}),0.021311,0.748538,6.970494
34,frozenset({ROSES REGENCY TEACUP AND SAUCER }),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.038848,0.732218,14.156468


In [40]:
print("All rules:", rules.shape)
print("Strong rules:", strong_rules.shape)

All rules: (184, 14)
Strong rules: (76, 14)


In [41]:
top_lift_rules = rules.sort_values(by="lift", ascending=False)
top_lift_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

,antecedents,consequents,support,confidence,lift
162,frozenset({WOODEN STAR CHRISTMAS SCANDINAVIAN}),frozenset({WOODEN HEART CHRISTMAS SCANDINAVIAN}),0.020423,0.768267,27.197263
161,frozenset({WOODEN HEART CHRISTMAS SCANDINAVIAN}),frozenset({WOODEN STAR CHRISTMAS SCANDINAVIAN}),0.020423,0.722986,27.197263
164,"frozenset({ROSES REGENCY TEACUP AND SAUCER , G...",frozenset({PINK REGENCY TEACUP AND SAUCER}),0.027305,0.702857,18.041001
167,frozenset({PINK REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , G...",0.027305,0.700855,18.041001
163,"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.027305,0.902752,17.453534
168,frozenset({GREEN REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",0.027305,0.527897,17.453534
166,frozenset({ROSES REGENCY TEACUP AND SAUCER }),"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",0.027305,0.514644,16.099612
165,"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.027305,0.854167,16.099612
31,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.031966,0.820513,15.863541
32,frozenset({GREEN REGENCY TEACUP AND SAUCER}),frozenset({PINK REGENCY TEACUP AND SAUCER}),0.031966,0.618026,15.863541


In [42]:
rules["antecedent_length"] = rules["antecedents"].apply(len)
rules["consequent_length"] = rules["consequents"].apply(len)

rules[["antecedent_length", "consequent_length"]].head()

,antecedent_length,consequent_length
163,2,1
165,2,1
31,1,1
171,2,1
176,2,1


In [43]:
rules.groupby(["antecedent_length", "consequent_length"]).size().reset_index(name="rule_count")

,antecedent_length,consequent_length,rule_count
0,1,1,163
1,1,2,9
2,2,1,12


In [44]:
final_rules = rules[
    (rules["confidence"] >= 0.5) &
    (rules["lift"] > 1.2)
].sort_values(by=["lift", "confidence"], ascending=False)

final_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(15)

,antecedents,consequents,support,confidence,lift
162,frozenset({WOODEN STAR CHRISTMAS SCANDINAVIAN}),frozenset({WOODEN HEART CHRISTMAS SCANDINAVIAN}),0.020423,0.768267,27.197263
161,frozenset({WOODEN HEART CHRISTMAS SCANDINAVIAN}),frozenset({WOODEN STAR CHRISTMAS SCANDINAVIAN}),0.020423,0.722986,27.197263
164,"frozenset({ROSES REGENCY TEACUP AND SAUCER , G...",frozenset({PINK REGENCY TEACUP AND SAUCER}),0.027305,0.702857,18.041001
167,frozenset({PINK REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , G...",0.027305,0.700855,18.041001
163,"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.027305,0.902752,17.453534
168,frozenset({GREEN REGENCY TEACUP AND SAUCER}),"frozenset({ROSES REGENCY TEACUP AND SAUCER , P...",0.027305,0.527897,17.453534
165,"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",frozenset({ROSES REGENCY TEACUP AND SAUCER }),0.027305,0.854167,16.099612
166,frozenset({ROSES REGENCY TEACUP AND SAUCER }),"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",0.027305,0.514644,16.099612
31,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.031966,0.820513,15.863541
32,frozenset({GREEN REGENCY TEACUP AND SAUCER}),frozenset({PINK REGENCY TEACUP AND SAUCER}),0.031966,0.618026,15.863541


## Summary

In this step, association rules were generated from the frequent itemsets identified in the previous stage in order to discover directional relationships between products. The rules were evaluated mainly using support, confidence, and lift.

A total of 184 association rules were generated. After filtering for stronger rules using confidence greater than or equal to 0.5 and lift greater than 1.2, a refined set of high-quality rules was obtained. These rules capture product relationships that are not only frequent enough to appear in the data, but also strong enough to suggest meaningful co-purchase behavior.

The rule structure analysis showed that most rules were one-to-one product relationships, while a smaller number involved one-to-two and two-to-one combinations. This indicates that the dataset contains both simple recommendation patterns and slightly more complex product associations.

The strongest rules by lift revealed highly related product combinations, especially among similar product variants and themed items. These findings are useful for recommendation systems, cross-selling strategies, and bundle planning in a retail setting.

In [45]:
final_rules.shape

(76, 16)

In [46]:
final_rules = final_rules.copy()

final_rules["antecedents_text"] = final_rules["antecedents"].apply(lambda x: ", ".join(list(x)))
final_rules["consequents_text"] = final_rules["consequents"].apply(lambda x: ", ".join(list(x)))

final_rules[["antecedents_text", "consequents_text", "support", "confidence", "lift"]].head(10)

,antecedents_text,consequents_text,support,confidence,lift
162,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,0.020423,0.768267,27.197263
161,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,0.020423,0.722986,27.197263
164,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",PINK REGENCY TEACUP AND SAUCER,0.027305,0.702857,18.041001
167,PINK REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",0.027305,0.700855,18.041001
163,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",GREEN REGENCY TEACUP AND SAUCER,0.027305,0.902752,17.453534
168,GREEN REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",0.027305,0.527897,17.453534
165,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,0.027305,0.854167,16.099612
166,ROSES REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",0.027305,0.514644,16.099612
31,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.031966,0.820513,15.863541
32,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.031966,0.618026,15.863541


In [47]:
recommendation_table = final_rules[
    ["antecedents_text", "consequents_text", "support", "confidence", "lift"]
].copy()

recommendation_table.columns = [
    "recommended_from",
    "recommended_to",
    "support",
    "confidence",
    "lift"
]

recommendation_table.head(10)

,recommended_from,recommended_to,support,confidence,lift
162,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,0.020423,0.768267,27.197263
161,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,0.020423,0.722986,27.197263
164,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",PINK REGENCY TEACUP AND SAUCER,0.027305,0.702857,18.041001
167,PINK REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",0.027305,0.700855,18.041001
163,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",GREEN REGENCY TEACUP AND SAUCER,0.027305,0.902752,17.453534
168,GREEN REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",0.027305,0.527897,17.453534
165,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,0.027305,0.854167,16.099612
166,ROSES REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",0.027305,0.514644,16.099612
31,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.031966,0.820513,15.863541
32,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.031966,0.618026,15.863541


In [48]:
recommendation_table["support"] = recommendation_table["support"].round(4)
recommendation_table["confidence"] = recommendation_table["confidence"].round(4)
recommendation_table["lift"] = recommendation_table["lift"].round(4)

recommendation_table.head(10)

,recommended_from,recommended_to,support,confidence,lift
162,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,0.0204,0.7683,27.1973
161,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,0.0204,0.7230,27.1973
164,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",PINK REGENCY TEACUP AND SAUCER,0.0273,0.7029,18.0410
167,PINK REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",0.0273,0.7009,18.0410
163,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",GREEN REGENCY TEACUP AND SAUCER,0.0273,0.9028,17.4535
168,GREEN REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",0.0273,0.5279,17.4535
165,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,0.0273,0.8542,16.0996
166,ROSES REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",0.0273,0.5146,16.0996
31,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.0320,0.8205,15.8635
32,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.0320,0.6180,15.8635


In [49]:
recommendation_table = recommendation_table.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

recommendation_table.head(15)

,recommended_from,recommended_to,support,confidence,lift
162,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,0.0204,0.7683,27.1973
161,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,0.0204,0.7230,27.1973
164,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",PINK REGENCY TEACUP AND SAUCER,0.0273,0.7029,18.0410
167,PINK REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",0.0273,0.7009,18.0410
163,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",GREEN REGENCY TEACUP AND SAUCER,0.0273,0.9028,17.4535
168,GREEN REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",0.0273,0.5279,17.4535
165,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,0.0273,0.8542,16.0996
166,ROSES REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",0.0273,0.5146,16.0996
31,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.0320,0.8205,15.8635
32,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.0320,0.6180,15.8635


In [50]:
recommendation_table["recommended_from"].value_counts().head(10)

recommended_from
STRAWBERRY CHARLOTTE BAG            4
DOTCOM POSTAGE                      4
JUMBO BAG WOODLAND ANIMALS          4
PINK REGENCY TEACUP AND SAUCER      3
GREEN REGENCY TEACUP AND SAUCER     3
ROSES REGENCY TEACUP AND SAUCER     3
WOODLAND CHARLOTTE BAG              3
CHARLOTTE BAG PINK POLKADOT         3
RED RETROSPOT CHARLOTTE BAG         3
CHARLOTTE BAG SUKI DESIGN           2
Name: count, dtype: int64

## Summary

In this step, the refined association rules were transformed into a cleaner recommendation table for business use. The antecedent and consequent itemsets were converted from frozenset format into readable text so that the product relationships could be interpreted more easily.

The final recommendation table was structured with five main fields: recommended product set, target recommendation set, support, confidence, and lift. The metrics were rounded for readability and the table was sorted by lift and confidence to prioritize the strongest recommendation patterns.

The results also showed that some products are linked to multiple recommendation outcomes, which reflects realistic retail behavior where a single product can be associated with more than one related product combination. This recommendation table can now be used in the next stage for SQL integration, dashboarding, or business reporting.

In [51]:
basket_df.shape

(530104, 9)

In [52]:
basket_df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'TotalPrice'],
      dtype='str')

In [53]:
sql_df = basket_df.copy()

sql_df["InvoiceDate"] = pd.to_datetime(sql_df["InvoiceDate"])

sql_df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
TotalPrice            float64
dtype: object

## SQL Table Design

The cleaned transaction data will be stored in SQL Server as a structured retail transactions table. This table contains invoice-level product records along with quantity, pricing, customer, and country information. It serves as the SQL base table for further business analysis such as top-selling products, country-wise sales, and monthly revenue trends.

In [54]:
sql_df.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00


```sql
CREATE DATABASE online_retail_project;
GO

USE online_retail_project;
GO

CREATE TABLE retail_transactions (
    InvoiceNo VARCHAR(20),
    StockCode VARCHAR(20),
    Description VARCHAR(MAX),
    Quantity INT,
    InvoiceDate DATETIME,
    UnitPrice DECIMAL(10,2),
    CustomerID VARCHAR(20),
    Country VARCHAR(100),
    TotalPrice DECIMAL(12,2)
);
GO

In [55]:
print("Rows to load into SQL table:", sql_df.shape[0])
print("Columns in SQL table:", sql_df.shape[1])

Rows to load into SQL table: 530104
Columns in SQL table: 9


In [56]:
sql_df.to_csv("retail_transactions_cleaned.csv", index=False)
print("CSV file exported successfully.")

CSV file exported successfully.


## Data Export for SQL Server

In this step, the cleaned transaction dataset was exported from Python as a CSV file so that it could be imported into SQL Server for further SQL- based analysis. This forms the bridge between the Python data preparation stage and the database layer of the project.t.

## Preparing Recommendation Rules for Power BI

In this step, the final association rule output is being reshaped into a cleaner recommendation table for Power BI. The rule fields are renamed into a more readable format, and the main recommendation metrics such as support, confidence, and lift are rounded for easier interpretation in visuals and tables.

This export focuses only on the recommendation rules table, since the broader KPI, trend, product presence, and product pair tables are being taken from SQL Server for the dashboard.

In [61]:
recommendation_rules = final_rules[
    [
        "antecedents_text",
        "consequents_text",
        "support",
        "confidence",
        "lift",
        "antecedent_length",
        "consequent_length"
    ]
].copy()

recommendation_rules.columns = [
    "recommended_from",
    "recommended_to",
    "support",
    "confidence",
    "lift",
    "antecedent_length",
    "consequent_length"
]

recommendation_rules["support"] = recommendation_rules["support"].round(4)
recommendation_rules["confidence"] = recommendation_rules["confidence"].round(4)
recommendation_rules["lift"] = recommendation_rules["lift"].round(4)

recommendation_rules.head(15)

,recommended_from,recommended_to,support,confidence,lift,antecedent_length,consequent_length
162,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,0.0204,0.7683,27.1973,1,1
161,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,0.0204,0.7230,27.1973,1,1
164,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",PINK REGENCY TEACUP AND SAUCER,0.0273,0.7029,18.0410,2,1
167,PINK REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , GREEN REGENC...",0.0273,0.7009,18.0410,1,2
163,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",GREEN REGENCY TEACUP AND SAUCER,0.0273,0.9028,17.4535,2,1
168,GREEN REGENCY TEACUP AND SAUCER,"ROSES REGENCY TEACUP AND SAUCER , PINK REGENCY...",0.0273,0.5279,17.4535,1,2
165,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",ROSES REGENCY TEACUP AND SAUCER,0.0273,0.8542,16.0996,2,1
166,ROSES REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY ...",0.0273,0.5146,16.0996,1,2
31,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.0320,0.8205,15.8635,1,1
32,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.0320,0.6180,15.8635,1,1


In [64]:
# Export Power BI Input Table

recommendation_rules.to_csv("recommendation_rules.csv", index=False)
print("Power BI input table exported successfully.")

Power BI input table exported successfully.


## Recommendation Rules Export Ready

The recommendation rules table has been exported as a separate CSV file for Power BI. This file is intended to support the recommendation-focused visuals in the dashboard, while the broader overview and basket-level summary tables are sourced from SQL Server.